# The Vernier dual-ring filter: measured spectra and the design space

Companion to web-app testbench **33 “Vernier dual-ring filter”**: two
cascaded add-drop rings with *slightly different* circumferences. Each ring
passes a comb of resonances at its own FSR; the cascade only transmits where
**both** combs align. Like a Vernier caliper, the two near-equal pitches beat
into a much longer effective pitch:

$$ \mathrm{FSR}_V \;=\; \frac{\mathrm{FSR}_1\,\mathrm{FSR}_2}
   {\left|\mathrm{FSR}_1-\mathrm{FSR}_2\right|} $$

— the trick every widely-tunable laser and FSR-extended WDM filter uses.
The catch: the closer the two FSRs (longer $\mathrm{FSR}_V$), the *less*
misaligned the neighbouring resonances are, so interstitial peaks leak
through. This notebook measures the stock testbench against the exact Airy
theory, then sweeps the design space to expose that trade.

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt

from photonflux.nb import Session

C0 = 299_792_458.0
s = Session()
bench = s.load_example("33_vernier_dual_ring")
wg1 = bench.instances["R1WA"]["settings"]
wg2 = bench.instances["R2WA"]["settings"]
k2 = bench.instances["R1C1"]["settings"]["coupling"]
NEFF, NG, LAM0 = wg1["neff"], wg1["n_group"], 1310.0
L1, L2 = 2 * wg1["length_m"], 2 * wg2["length_m"]     # two half-rings each
P_IN = bench["LAS.power"]
m1 = NEFF * L1 / (LAM0 * 1e-9)
m2 = NEFF * L2 / (LAM0 * 1e-9)
print(f"ring 1: L = {L1 * 1e6:.2f} µm (order m = {m1:.1f}) | "
      f"ring 2: L = {L2 * 1e6:.2f} µm (m = {m2:.1f}) | κ² = {k2}")

Both orders are integers at 1310 nm — the testbench lengths are chosen so
the two combs *coincide* exactly at band centre. On paper:

$$ \mathrm{FSR}_i = \frac{\lambda^2}{n_g L_i}, \qquad
   \text{finesse} \approx \frac{\pi\sqrt{q}}{1-q},\quad
   q = (1-\kappa^2)\,a,\quad a = 10^{-\alpha_{dB} L/20} $$

and the drop-port lineshape is the exact **Airy response**
$ T(\varphi) \propto 1/(1 + F\sin^2(\varphi/2)) $ with
$F = 4q/(1-q)^2$, $\varphi = 2\pi\,\delta\lambda/\mathrm{FSR}$.

In [ ]:
def ring_params(L, k2_):
    fsr = LAM0 ** 2 * 1e-18 / (NG * L) * 1e9            # nm
    a = 10 ** (-wg1["loss_dB_cm"] * (L * 100) / 20)     # field, one round trip
    q = (1 - k2_) * a
    fin = np.pi * np.sqrt(q) / (1 - q)
    il = (k2_ ** 2 * np.sqrt(a)) / (1 - q) ** 2         # drop peak transmission
    return {"fsr": fsr, "q": q, "F": 4 * q / (1 - q) ** 2,
            "finesse": fin, "fwhm": fsr / fin, "il": il}


r1, r2 = ring_params(L1, k2), ring_params(L2, k2)
fsr_v = r1["fsr"] * r2["fsr"] / abs(r1["fsr"] - r2["fsr"])
print(f"FSR1 = {r1['fsr']:.3f} nm, FSR2 = {r2['fsr']:.3f} nm "
      f"-> Vernier FSR = {fsr_v:.1f} nm")
print(f"finesse = {r1['finesse']:.1f}, FWHM = {r1['fwhm'] * 1e3:.0f} pm, "
      f"drop IL = {10 * np.log10(r1['il']):.2f} dB per ring")


def airy(dlam, rp):
    """Normalised drop-port power response at dlam [nm] off resonance."""
    return 1 / (1 + rp["F"] * np.sin(np.pi * dlam / rp["fsr"]) ** 2)


def peaks(x, y, thresh, min_sep):
    idx = [i for i in range(1, len(y) - 1)
           if y[i] >= y[i - 1] and y[i] > y[i + 1] and y[i] > thresh]
    idx.sort(key=lambda i: -y[i])
    keep = []
    for i in idx:
        if all(abs(x[i] - x[j]) > min_sep for j in keep):
            keep.append(i)
    return np.array(sorted(keep), int)

## 1. The stock filter

The stored analysis sweeps `wavelength_nm` on every instance at once
(`instance="*"` — laser and waveguides together), 1300 → 1320.5 nm.
`drop1` is ring 1's drop port (which feeds ring 2), `drop2` is the
composite Vernier output.

In [ ]:
res = s.run(schematic=bench)
wl = res.x
d1, d2 = res["drop1"], res["drop2"]         # mW
db1 = 10 * np.log10(np.maximum(d1, 1e-12) / (P_IN * 1e3))
db2 = 10 * np.log10(np.maximum(d2, 1e-12) / (P_IN * 1e3))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(9.5, 5.4), sharex=True)
a1.plot(wl, db1, lw=.8, color="C0"), a1.set_ylabel("drop1 [dB]")
a1.set_title("ring 1 alone: a comb at FSR$_1$")
a2.plot(wl, db2, lw=.8, color="C1"), a2.set_ylabel("drop2 [dB]")
a2.set_title("the cascade: one Vernier coincidence, interstitials suppressed")
a2.set_xlabel("wavelength [nm]")
for a in (a1, a2):
    a.grid(alpha=.3), a.set_ylim(-55, 2)
fig.tight_layout()

In [ ]:
pk1 = peaks(wl, d1, thresh=d1.max() * 0.03, min_sep=r1["fsr"] / 2)
meas_fsr1 = np.diff(wl[pk1])
loc = LAM0 ** 2 * 1e-18 / (NG * L1) * 1e9   # at 1310; scale each gap by λ²
pred = ((wl[pk1][:-1] + wl[pk1][1:]) / 2 / LAM0) ** 2 * loc
print(f"FSR1 measured {meas_fsr1.mean():.4f} nm (theory {loc:.4f} at 1310, "
      f"max dev {np.max(np.abs(meas_fsr1 / pred - 1)) * 100:.1f}%)")
assert np.max(np.abs(meas_fsr1 / pred - 1)) < 0.02

main = wl[np.argmax(d2)]
print(f"coincidence at {main:.4f} nm (designed: 1310.0000)")
assert abs(main - 1310.0) < 0.02

## 2. Interstitial suppression, line by line, vs exact Airy theory

At the $k$-th ring-1 resonance away from the coincidence, ring 2's nearest
resonance is misaligned by $\delta_k = k\,\mathrm{FSR}_1 \bmod \mathrm{FSR}_2$
(folded to $\pm\mathrm{FSR}_2/2$). The composite peak there is the maximum
of the product of the two Airy lineshapes, one shifted by $\delta_k$ —
no fitted parameters, everything from $\kappa^2$ and the loss.

In [ ]:
def fold(x, per):
    return (x + per / 2) % per - per / 2


def supp_pred(delta):
    """Peak of Airy1(x)·Airy2(x - delta), in dB re the aligned case."""
    x = np.linspace(-abs(delta), abs(delta), 801) if delta else np.array([0.])
    prod = airy(x, r1) * airy(x - delta, r2)   # both peak-normalised
    return 10 * np.log10(prod.max())


pk2 = peaks(wl, d2, thresh=d2.max() * 1e-4, min_sep=r1["fsr"] / 2)
main_db = db2.max()
ks = np.round((wl[pk2] - main) / (((wl[pk2] + main) / 2 / LAM0) ** 2 * loc))
meas_supp = db2[pk2] - main_db
pred_supp = np.array([supp_pred(fold(k * r1["fsr"], r2["fsr"])) for k in ks])

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(wl[pk2], meas_supp, "o", label="testbench 33 (drop2 peaks)")
ax.plot(wl[pk2], pred_supp, "x--", label="Airy product, zero fit parameters")
ax.set_xlabel("wavelength [nm]"), ax.set_ylabel("peak level re coincidence [dB]")
ax.legend(fontsize=8), ax.grid(alpha=.3)
rms = float(np.sqrt(np.mean((meas_supp - pred_supp) ** 2)))
ax.set_title(f"interstitial suppression: RMS error {rms:.2f} dB")
fig.tight_layout()
print(f"worst interstitial: measured {sorted(meas_supp)[-2]:.1f} dB")
assert rms < 2.5, rms

## 3. Linewidth and insertion loss at the coincidence

A zoomed sweep (±0.5 nm, 0.5 pm/point) resolves the composite passband.
Two aligned Airy peaks multiply, so the cascade is *narrower* than one
ring — the numeric product gives the exact factor (≈ 0.64 for identical
Lorentzian-limit rings).

In [ ]:
def fwhm_of(x, p):
    i = int(np.argmax(p))
    h = p[i] / 2
    l = i
    while l > 0 and p[l] > h:
        l -= 1
    r = i
    while r < len(p) - 1 and p[r] > h:
        r += 1
    xl = x[l] + (h - p[l]) * (x[l + 1] - x[l]) / (p[l + 1] - p[l])
    xr = x[r - 1] + (h - p[r - 1]) * (x[r] - x[r - 1]) / (p[r] - p[r - 1])
    return xr - xl


ZOOM = {"mode": "dcsweep", "instance": "*", "param": "wavelength_nm",
        "start": 1309.5, "stop": 1310.5, "points": 2001}
z = s.run(dict(ZOOM), schematic=bench)
xz = np.linspace(-r1["fsr"] / 2, r1["fsr"] / 2, 20001)
comp = airy(xz, r1) * airy(xz, r2)
fw_pred = fwhm_of(xz, comp)
fw_meas = fwhm_of(z.x, z["drop2"])
il_meas = 10 * np.log10(z["drop2"].max() / (P_IN * 1e3))
il_pred = 10 * np.log10(r1["il"] * r2["il"])
print(f"composite FWHM: measured {fw_meas * 1e3:.1f} pm, Airy² {fw_pred * 1e3:.1f} pm"
      f" (single ring {r1['fwhm'] * 1e3:.1f} pm)")
print(f"insertion loss: measured {il_meas:.2f} dB, theory {il_pred:.2f} dB")
assert abs(fw_meas / fw_pred - 1) < 0.10
assert abs(il_meas - il_pred) < 0.5

## 4. Design space I — how far apart should the rings be?

The single design knob behind the Vernier is the **order difference**
$\Delta m = m_1 - m_2$. Sweep it (adjusting ring 2's length, snapped so a
resonance stays at 1310 nm) and measure the two quantities a filter
designer trades:

* $\mathrm{FSR}_V = \mathrm{FSR}_1 \cdot m_2/\Delta m$ — how far the next
  coincidence moves away,
* the **worst interstitial** in the band — how well everything between is
  rejected ($\delta_1 = |\mathrm{FSR}_2-\mathrm{FSR}_1|$ shrinks as
  $\Delta m$ shrinks).

One curve, opposite slopes: the hyperbola every Vernier design lives on.

In [ ]:
lam_res_m = LAM0 * 1e-9 / NEFF                 # length quantum per order
m2_sweep = [300, 315, 330, 345, 356]
rows = []
for m2_i in m2_sweep:
    L2i = m2_i * lam_res_m
    b = copy.deepcopy(bench.doc)
    for ref in ("R2WA", "R2WB"):
        b["schematic"]["instances"][ref]["settings"]["length_m"] = L2i / 2
    r = s.run(bench.analysis, schematic=b)
    dbi = 10 * np.log10(np.maximum(r["drop2"], 1e-12) / (P_IN * 1e3))
    p = peaks(r.x, r["drop2"], thresh=r["drop2"].max() * 1e-4,
              min_sep=r1["fsr"] / 2)
    worst = np.sort(dbi[p] - dbi.max())[-2] if len(p) > 1 else -np.inf
    r2i = ring_params(L2i, k2)
    d1i = abs(r2i["fsr"] - r1["fsr"])
    rows.append({"dm": m1 - m2_i,
                 "fsr_v": r1["fsr"] * r2i["fsr"] / d1i,
                 "worst": worst, "pred": supp_pred(d1i)})
    print(f"Δm = {rows[-1]['dm']:3.0f}: FSR_V = {rows[-1]['fsr_v']:6.1f} nm, "
          f"worst interstitial {worst:6.1f} dB (Airy: {rows[-1]['pred']:6.1f})")

dm = np.array([r["dm"] for r in rows])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dm, [r["worst"] for r in rows], "o-", label="measured worst interstitial")
ax.plot(dm, [r["pred"] for r in rows], "x--", label="Airy product prediction")
ax2 = ax.twinx()
ax2.plot(dm, [r["fsr_v"] for r in rows], "s-", color="tab:green")
ax2.set_ylabel("Vernier FSR [nm]", color="tab:green")
ax.set_xlabel("order difference Δm"), ax.set_ylabel("worst interstitial [dB]")
ax.legend(fontsize=8, loc="lower left"), ax.grid(alpha=.3)
ax.set_title("extend the FSR, lose the rejection")
fig.tight_layout()
for r in rows:
    assert abs(r["worst"] - r["pred"]) < 3.0, r

## 5. Design space II — coupling sets the linewidth/loss/rejection knot

Same geometry, sweep $\kappa^2$ on all four couplers. Lower coupling means
higher finesse: sharper channels *and* deeper interstitial rejection — but
the resonator stores light longer, so the fixed 5 dB/cm loss bites harder
and the drop-port insertion loss grows. All three curves against the same
zero-parameter Airy model.

In [ ]:
k2_sweep = [0.03, 0.05, 0.1, 0.2, 0.3]
meas = {"fwhm": [], "il": [], "worst": []}
pred = {"fwhm": [], "il": [], "worst": []}
for k2_i in k2_sweep:
    b = copy.deepcopy(bench.doc)
    for ref in ("R1C1", "R1C2", "R2C1", "R2C2"):
        b["schematic"]["instances"][ref]["settings"]["coupling"] = k2_i
    z = s.run(dict(ZOOM), schematic=b)
    f = s.run(bench.analysis, schematic=b)
    r1i, r2i = ring_params(L1, k2_i), ring_params(L2, k2_i)
    meas["fwhm"].append(fwhm_of(z.x, z["drop2"]) * 1e3)
    meas["il"].append(10 * np.log10(z["drop2"].max() / (P_IN * 1e3)))
    dbi = 10 * np.log10(np.maximum(f["drop2"], 1e-12) / (P_IN * 1e3))
    p = peaks(f.x, f["drop2"], thresh=f["drop2"].max() * 1e-5,
              min_sep=r1["fsr"] / 2)
    meas["worst"].append(np.sort(dbi[p] - dbi.max())[-2])
    comp = airy(xz, r1i) * airy(xz, r2i)
    pred["fwhm"].append(fwhm_of(xz, comp) * 1e3)
    pred["il"].append(10 * np.log10(r1i["il"] * r2i["il"]))
    d1i = abs(r2i["fsr"] - r1i["fsr"])
    x = np.linspace(-d1i, d1i, 801)
    pk = (airy(x, r1i) * airy(x - d1i, r2i)).max()
    pred["worst"].append(10 * np.log10(pk))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, key, unit in zip(axes, ("fwhm", "il", "worst"),
                         ("FWHM [pm]", "insertion loss [dB]",
                          "worst interstitial [dB]")):
    ax.plot(k2_sweep, meas[key], "o-", label="measured")
    ax.plot(k2_sweep, pred[key], "x--", label="Airy theory")
    ax.set_xlabel("κ² per coupler"), ax.set_ylabel(unit)
    ax.grid(alpha=.3), ax.legend(fontsize=8)
axes[0].set_yscale("log")
fig.tight_layout()
np.testing.assert_allclose(meas["fwhm"], pred["fwhm"], rtol=0.15)
assert np.max(np.abs(np.array(meas["il"]) - pred["il"])) < 0.7
assert np.max(np.abs(np.array(meas["worst"]) - np.array(pred["worst"]))) < 3.0

---
**Takeaways.** The testbench matches the zero-fit-parameter Airy cascade —
FSRs to 2 %, insertion loss to a fraction of a dB, interstitial suppression
to a couple of dB across both design sweeps. The two charts above *are* the
Vernier design procedure: pick Δm from the FSR you must cover and the
rejection you can live with, then pick κ² from the linewidth/loss budget.

**Things to try**

* Detune ring 2 by one part in 10⁵ (`bench["R2WA.length_m"] *= 1.00001`,
  same for `R2WB`) — the coincidence walks off 1310 nm by a full FSR₁:
  that's the *thermal tuning* mechanism of a Vernier laser.
* Drop the loss to 1 dB/cm and re-run section 5 — watch the insertion loss
  collapse while the rejection deepens: loss is what caps this filter.
* Testbench 37 (“Vernier laser mode hop”) puts this exact filter inside a
  laser cavity.